In [1]:
import os
import numpy as np
import pandas as pd
import scipy
import anndata
import scanpy as sc
import pybiomart
import scvi
import torch
import random
import seaborn as sns

In [2]:
cwd = os.getcwd()
cwd

'/home/merlin/Documents/Programming/FigureOneLab/trevino'

In [3]:
# Load metadata into pandas dataframe
meta = pd.read_csv(cwd+'/data/GSE162170_rna_cell_metadata.txt.gz', compression='gzip', sep='\t')
meta.index = meta['Cell.ID']  # Cell ID's as row names
meta.index.name = None  # Clean up index names
meta

,Cell.ID,Sample.ID,Age,Tissue.ID,Sample.Type,Assay,Batch,seurat_clusters,RNA.Counts,RNA.Features,...,Cell.Barcode,DF_pANN,DF_classification,DF_pANN_quantile,Spliced.Counts,Spliced.Features,Unspliced.Counts,Unspliced.Features,Ambiguous.Counts,Ambiguous.Features
hft_w20_p3_r1_AAACCCAAGCTGCGAA,hft_w20_p3_r1_AAACCCAAGCTGCGAA,hft_w20_p3_r1,pcw20,HFT3,HFT,RNA v3,b2019_06,c16,1397,677,...,AAACCCAAGCTGCGAA,0.086339,Singlet,0.356997,1063,544,67,54,94,69
hft_w20_p3_r1_AAACCCAAGGTAGTAT,hft_w20_p3_r1_AAACCCAAGGTAGTAT,hft_w20_p3_r1,pcw20,HFT3,HFT,RNA v3,b2019_06,c11,14338,4301,...,AAACCCAAGGTAGTAT,0.325683,Singlet,0.821429,10339,3514,5437,2526,1431,669
hft_w20_p3_r1_AAACCCACAACTCCAA,hft_w20_p3_r1_AAACCCACAACTCCAA,hft_w20_p3_r1,pcw20,HFT3,HFT,RNA v3,b2019_06,c17,9260,3481,...,AAACCCACAACTCCAA,0.397814,Doublet,0.984402,6494,2701,6860,2515,1095,669
hft_w20_p3_r1_AAACCCACATAGTCAC,hft_w20_p3_r1_AAACCCACATAGTCAC,hft_w20_p3_r1,pcw20,HFT3,HFT,RNA v3,b2019_06,c0,4025,1969,...,AAACCCACATAGTCAC,0.076503,Singlet,0.314723,2655,1475,5875,2058,634,377
hft_w20_p3_r1_AAACCCAGTACAGGTG,hft_w20_p3_r1_AAACCCAGTACAGGTG,hft_w20_p3_r1,pcw20,HFT3,HFT,RNA v3,b2019_06,c4,7131,2930,...,AAACCCAGTACAGGTG,0.239344,Singlet,0.746356,5008,2228,6026,2106,909,556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
hft_w16_p7_r2_TTTGTTGCAGCACCCA,hft_w16_p7_r2_TTTGTTGCAGCACCCA,hft_w16_p7_r2,pcw16,HFT7,HFT,RNA NG,b2020_03,c6,8532,3497,...,TTTGTTGCAGCACCCA,0.389488,Doublet,0.961542,6226,2730,4194,2059,973,612
hft_w16_p7_r2_TTTGTTGCAGGCTACC,hft_w16_p7_r2_TTTGTTGCAGGCTACC,hft_w16_p7_r2,pcw16,HFT7,HFT,RNA NG,b2020_03,c2,6689,2393,...,TTTGTTGCAGGCTACC,0.210916,Singlet,0.663941,4757,1877,3039,1362,677,396
hft_w16_p7_r2_TTTGTTGGTCGCTTAA,hft_w16_p7_r2_TTTGTTGGTCGCTTAA,hft_w16_p7_r2,pcw16,HFT7,HFT,RNA NG,b2020_03,c2,3865,1799,...,TTTGTTGGTCGCTTAA,0.074798,Singlet,0.306766,2808,1409,1436,832,431,273
hft_w16_p7_r2_TTTGTTGGTCGTACAT,hft_w16_p7_r2_TTTGTTGGTCGTACAT,hft_w16_p7_r2,pcw16,HFT7,HFT,RNA NG,b2020_03,c2,5293,2365,...,TTTGTTGGTCGTACAT,0.208895,Singlet,0.660796,3660,1760,6166,2211,672,416


In [4]:
%%time
# Old approach was causing OOM issues due to large file size and duplication in transpose function
# counts = pd.read_csv(cwd+'/data/GSE162170_rna_counts.tsv.gz', compression='gzip', sep='\t')
# counts = counts.transpose()
# counts

# Using built in scanpy function directly
adata = sc.read_csv(cwd+'/data/GSE162170_rna_counts.tsv.gz', delimiter='\t')

CPU times: user 1min 17s, sys: 7.5 s, total: 1min 24s
Wall time: 1min 25s


In [5]:
adata = adata.transpose()
adata

AnnData object with n_obs × n_vars = 57868 × 33355

In [6]:
%%time
# adata = anndata.AnnData(X=counts,
#                         obs=meta,
#                         var=counts.columns.to_frame())

adata.obs = meta.loc[adata.obs.index]  # Align by cell ID
adata

CPU times: user 19.9 ms, sys: 997 µs, total: 20.9 ms
Wall time: 20.4 ms


AnnData object with n_obs × n_vars = 57868 × 33355
    obs: 'Cell.ID', 'Sample.ID', 'Age', 'Tissue.ID', 'Sample.Type', 'Assay', 'Batch', 'seurat_clusters', 'RNA.Counts', 'RNA.Features', 'Percent.MT', 'Percent.Ribo', 'Cell.Barcode', 'DF_pANN', 'DF_classification', 'DF_pANN_quantile', 'Spliced.Counts', 'Spliced.Features', 'Unspliced.Counts', 'Unspliced.Features', 'Ambiguous.Counts', 'Ambiguous.Features'

In [7]:
print("adata.var columns:", adata.var.columns.tolist())
print("adata.var head:")
print(adata.var.head())

adata.var columns: []
adata.var head:
Empty DataFrame
Columns: []
Index: [ENSG00000243485, ENSG00000237613, ENSG00000186092, ENSG00000238009, ENSG00000239945]


In [8]:
%%time
a = sc.queries.biomart_annotations('hsapiens', ['ensembl_gene_id','hgnc_symbol'])
b = dict(zip(a['ensembl_gene_id'], a['hgnc_symbol']))
adata.var['hgnc_symbol'] = adata.var.index.map(b)
adata.var.index = adata.var['hgnc_symbol']
adata.var.index.name = None
adata.var.drop(columns=['hgnc_symbol'], inplace=True)
adata.var

CPU times: user 219 ms, sys: 22.9 ms, total: 242 ms
Wall time: 29.5 s


""
MIR1302-2HG
FAM138A
OR4F5
NaN
NaN
...
MT-ND4L
MT-ND4
MT-ND5
MT-ND6


In [9]:
adata.var_names_make_unique

<bound method AnnData.var_names_make_unique of AnnData object with n_obs × n_vars = 57868 × 33355
    obs: 'Cell.ID', 'Sample.ID', 'Age', 'Tissue.ID', 'Sample.Type', 'Assay', 'Batch', 'seurat_clusters', 'RNA.Counts', 'RNA.Features', 'Percent.MT', 'Percent.Ribo', 'Cell.Barcode', 'DF_pANN', 'DF_classification', 'DF_pANN_quantile', 'Spliced.Counts', 'Spliced.Features', 'Unspliced.Counts', 'Unspliced.Features', 'Ambiguous.Counts', 'Ambiguous.Features'>

In [ ]:
a = ~adata.var.index.isnull()
# Also OOM error while running due to duplicate creation
# adata = adata[:,a].copy()
# Instead subset using direct approach to reduce memory usage

adata = adata[:, a]

In [ ]:
adata

In [ ]:
%%time
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.filter_cells(adata, min_genes=200)

In [ ]:
#sc.pp.subsample(adata, n_obs=40000, random_state=0, copy=False)

In [ ]:
%%time
adata.X = scipy.sparse.csr_matrix(adata.X.copy())
adata.layers['counts'] = scipy.sparse.csr_matrix(adata.X.copy())
#adata.layers['log2_counts'] = sc.pp.log1p(adata.layers['counts'].copy(), base=2)

In [ ]:
%%time
random.seed(17)
scvi.model.SCVI.setup_anndata(adata, layer='counts', batch_key='Sample.ID')
scvi_model = scvi.model.SCVI(adata, n_layers=2, n_latent=30, n_hidden=128, gene_likelihood='nb')
scvi_model.train()

In [ ]:
%%time
random.seed(17)
adata.obsm['X_scvi'] = scvi_model.get_latent_representation()
adata.layers['counts_scvi'] = scvi_model.get_normalized_expression(library_size=10000)
#adata.layers['log2_counts_scvi'] = sc.pp.log1p(adata.layers['counts_scvi'].copy(), base=2)

In [ ]:
%%time
sc.pp.neighbors(adata, use_rep='X_scvi', key_added='neighbors_scvi', n_neighbors=20)
sc.tl.leiden(adata, neighbors_key='neighbors_scvi', key_added='leiden_scvi', resolution=3)
sc.tl.umap(adata, neighbors_key='neighbors_scvi')
sc.pl.umap(adata, color=['leiden_scvi'], legend_loc='on data')

In [ ]:
%%time
sc.pp.neighbors(adata, use_rep='X_scvi', key_added='neighbors_scvi', n_neighbors=20)
sc.tl.leiden(adata, neighbors_key='neighbors_scvi', key_added='leiden_scvi', resolution=3)
sc.tl.umap(adata, neighbors_key='neighbors_scvi')
sc.pl.umap(adata, color=['leiden_scvi'], legend_loc='on data')

In [ ]:
%%time
adata.write(cwd+'/outs/231226_trevino_rna_scvi.h5ad')